In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CAT = 'northbridge'
B_CU = f'{CAT}.bronze.customers'
DIM  = f'{CAT}.silver.dim_customer'
DQ   = f'{CAT}.ops.dq_results'

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {DIM} (
  customer_id STRING, full_name STRING, phone STRING, city STRING,
  account_status STRING, valid_from TIMESTAMP, valid_to TIMESTAMP, is_current BOOLEAN)""")

In [0]:
def load_dim_customer(day):
    # 1) today's incoming rows, typed, one row per customer
    inc = (spark.table(B_CU)
        .withColumn('day', F.col('day').cast('int')).filter(F.col('day') == day)
        .withColumn('updated_at', F.expr('try_cast(updated_at as timestamp)'))
        .filter(F.col('customer_id').isNotNull() & F.col('updated_at').isNotNull()))
    w = Window.partitionBy('customer_id').orderBy(F.col('updated_at').desc(), F.col('_ingest_ts').desc())
    inc = (inc.withColumn('rn', F.row_number().over(w)).filter('rn = 1')
              .select('customer_id', 'full_name', 'phone', 'city', 'account_status', 'updated_at'))

    # 2) the current version of each customer already in the dimension
    cur = (spark.table(DIM).filter('is_current')
        .select(F.col('customer_id').alias('c_id'), F.col('full_name').alias('c_name'),
                F.col('phone').alias('c_phone'), F.col('city').alias('c_city'),
                F.col('account_status').alias('c_status'), F.col('valid_from').alias('c_from')))

    # 3) keep only genuinely new customers, or real changes that are NEWER than the current version
    differs = (~F.col('full_name').eqNullSafe(F.col('c_name'))
             | ~F.col('phone').eqNullSafe(F.col('c_phone'))
             | ~F.col('city').eqNullSafe(F.col('c_city'))
             | ~F.col('account_status').eqNullSafe(F.col('c_status')))
    changed = (inc.join(cur, inc.customer_id == cur.c_id, 'left')
        .filter(F.col('c_id').isNull() | ((F.col('updated_at') > F.col('c_from')) & differs)))

    # 4) the "union twice" trick: one copy with a key (closes the old row), one with a null key (inserts the new row)
    cols = ['customer_id', 'full_name', 'phone', 'city', 'account_status', 'updated_at']
    close_rows = changed.filter(F.col('c_id').isNotNull()).select(*cols, F.col('customer_id').alias('merge_key'))
    new_rows   = changed.select(*cols, F.lit(None).cast('string').alias('merge_key'))
    close_rows.unionByName(new_rows).createOrReplaceTempView('scd_stage')

    spark.sql(f"""
      MERGE INTO {DIM} t USING scd_stage s
      ON t.customer_id = s.merge_key AND t.is_current = true
      WHEN MATCHED THEN UPDATE SET t.is_current = false, t.valid_to = s.updated_at
      WHEN NOT MATCHED THEN INSERT (customer_id, full_name, phone, city, account_status,
                                    valid_from, valid_to, is_current)
        VALUES (s.customer_id, s.full_name, s.phone, s.city, s.account_status,
                s.updated_at, TIMESTAMP'9999-12-31 00:00:00', true)""")
    print('dim_customer loaded for day', day)

In [0]:
load_dim_customer(1)
print(spark.table(DIM).count())      # expect 6,353,307

load_dim_customer(15)
print(spark.table(DIM).count())      # expect 6,353,307 + the number of day-15 rows

In [0]:
display(spark.sql(f"""
  SELECT count(*) AS total_rows,
         sum(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows,
         sum(CASE WHEN NOT is_current THEN 1 ELSE 0 END) AS closed_rows,
         count(DISTINCT customer_id) AS customers
  FROM {DIM}"""))

n15 = spark.table(B_CU).withColumn('day', F.col('day').cast('int')).filter('day = 15').count()
print('day-15 rows in bronze:', n15)

In [0]:
bad = spark.sql(f"""SELECT customer_id FROM {DIM} WHERE is_current
                    GROUP BY customer_id HAVING count(*) > 1""").count()
print('customers with more than one current row:', bad)    # must be 0

In [0]:
display(spark.sql(f"""
  SELECT * FROM {DIM}
  WHERE customer_id IN (SELECT customer_id FROM {DIM} WHERE NOT is_current LIMIT 1)
  ORDER BY valid_from"""))

In [0]:
before = spark.table(DIM).count()
load_dim_customer(15)
load_dim_customer(1)        # replaying the OLD day after the newer one
after = spark.table(DIM).count()
print(before, after, before == after)     # must be True

In [0]:
def log_dq(name, ok, n, notes=''):
    (spark.createDataFrame([(name, 'PASS' if ok else 'FAIL', n, notes)],
         'check_name string, status string, row_count long, notes string')
      .withColumn('run_ts', F.current_timestamp())
      .select('check_name', 'run_ts', 'status', 'row_count', 'notes')
      .write.mode('append').saveAsTable(DQ))

d = spark.table(DIM)
closed = d.filter('NOT is_current').count()
log_dq('dim_customer_one_current_per_customer', bad == 0, d.filter('is_current').count())
log_dq('dim_customer_closed_rows_eq_day15_changes', closed == n15, closed)
display(spark.table(DQ).orderBy(F.col('run_ts').desc()).limit(6))

In [0]:
try:
    spark.sql(f"ALTER TABLE {DIM} ALTER COLUMN phone SET MASK {CAT}.silver.mask_phone")
    print('column mask attached')
except Exception as e:
    print('mask not available here:', str(e)[:300])

spark.sql(f"""CREATE OR REPLACE VIEW {CAT}.silver.dim_customer_masked AS
  SELECT customer_id, full_name, {CAT}.silver.mask_phone(phone) AS phone,
         city, account_status, valid_from, valid_to, is_current FROM {DIM}""")

display(spark.sql(f"SELECT customer_id, phone FROM {CAT}.silver.dim_customer_masked LIMIT 3"))